# Setup

- macOS: Run 'brew install poppler'
- Ubuntu/Debian: Run 'apt-get install poppler-utils'
- Windows: Install from https://github.com/oschwartz10612/poppler-windows/releases/ and add to PATH

In [ ]:
# !pip install typhoon-ocr==0.4.1
# !pip install python-dotenv
# !pip install lxml
# !pip install html5lib

In [ ]:
from dotenv import load_dotenv
import os
import pandas as pd
from typhoon_ocr import ocr_document
from io import StringIO
import re
from utils import *

load_dotenv()

In [ ]:
files = [
    # ================ DISTRICT ==================
    {
        "path": "district/advance/5-17 เขต 1 จ.บุรีรัมย์", 
        "pages": 16
    },
    {
        "path": "district/ontime/อำเภอบ้านด่าน/ตำบลบ้านด่าน",
        "pages": 40
    },
    {
        "path": "district/ontime/อำเภอบ้านด่าน/ตำบลปราสาท",
        "pages": 32
    },
    {
        "path": "district/ontime/อำเภอเมืองบุรีรัมย์/5ทับ18 ตำบลถลุงเหล็ก",
        "pages": 28
    },
    {
        "path": "district/ontime/อำเภอเมืองบุรีรัมย์/ตำบลกระสัง",
        "pages": 28
    },
    {
        "path": "district/ontime/อำเภอเมืองบุรีรัมย์/ตำบลกลันทา",
        "pages": 26
    },
    {
        "path": "district/ontime/อำเภอเมืองบุรีรัมย์/ตำบลชุมเห็ด",
        "pages": 64
    },
    {
        "path": "district/ontime/อำเภอเมืองบุรีรัมย์/ตำบลในเมือง",
        "pages": 62
    },
    {
        "path": "district/ontime/อำเภอเมืองบุรีรัมย์/ตำบลบัวทอง",
        "pages": 28
    },
    {
        "path": "district/ontime/อำเภอเมืองบุรีรัมย์/ตำบลบ้านบัว",
        "pages": 38
    },
    {
        "path": "district/ontime/อำเภอเมืองบุรีรัมย์/ตำบลบ้านยาง",
        "pages": 44
    },
    {
        "path": "district/ontime/อำเภอเมืองบุรีรัมย์/ตำบลพระครู",
        "pages": 22
    },
    {
        "path": "district/ontime/อำเภอเมืองบุรีรัมย์/ตำบลลุมปุ๊ก",
        "pages": 38,
    },
    {
        "path": "district/ontime/อำเภอเมืองบุรีรัมย์/ตำบลสะแกโพรง",
        "pages": 48,
    },
    {
        "path": "district/ontime/อำเภอเมืองบุรีรัมย์/ตำบลหนองตาด",
        "pages": 44
    },
    
    # ================ PARTYLIST ==================
    {
        "path": "partylist/advance/5-17(บช)",
        "pages": 32,
    },
    {
        "path": "partylist/ontime/อำเภอบ้านด่าน/ตำบลบ้านด่าน",
        "pages": 80
    },
    {
        "path": "partylist/ontime/อำเภอบ้านด่าน/ตำบลปราสาท",
        "pages": 64
    },
    {
        "path": "partylist/ontime/อำเภอเมืองบุรีรัมย์/ตำบลกระสัง",
        "pages": 56
    },
    {
        "path": "partylist/ontime/อำเภอเมืองบุรีรัมย์/ตำบลกลันทา",
        "pages": 52
    },
    {
        "path": "partylist/ontime/อำเภอเมืองบุรีรัมย์/ตำบลชุมเห็ด",
        "pages": 128
    },
    {
        "path": "partylist/ontime/อำเภอเมืองบุรีรัมย์/ตำบลถลุงเหล็ก",
        "pages": 56
    },
    {
        "path": "partylist/ontime/อำเภอเมืองบุรีรัมย์/ตำบลในเมือง",
        "pages": 124
    },
    {
        "path": "partylist/ontime/อำเภอเมืองบุรีรัมย์/ตำบลบัวทอง",
        "pages": 56
    },
    {
        "path": "partylist/ontime/อำเภอเมืองบุรีรัมย์/ตำบลบ้านบัว",
        "pages": 76
    },
    {
        "path": "partylist/ontime/อำเภอเมืองบุรีรัมย์/ตำบลบ้านยาง",
        "pages": 88
    },
    {
        "path": "partylist/ontime/อำเภอเมืองบุรีรัมย์/ตำบลพระครู",
        "pages": 44
    },
    {
        "path": "partylist/ontime/อำเภอเมืองบุรีรัมย์/ตำบลลุมปุ๊ก",
        "pages": 76
    },
    {
        "path": "partylist/ontime/อำเภอเมืองบุรีรัมย์/ตำบลสะแกโพรง",
        "pages": 96
    },
    {
        "path": "partylist/ontime/อำเภอเมืองบุรีรัมย์/ตำบลหนองตาด",
        "pages": 88
    },
    
    # ================ REFERENDUM ==================
    {
        "path": "referendum/outside/อส 4-7 นอกเขตออกเสียง",
        "pages": 4
    },
    {
        "path": "referendum/inside/อำเภอบ้านด่าน/ตำบลบ้านด่าน",
        "pages": 40
    },
    {
        "path": "referendum/inside/อำเภอบ้านด่าน/ตำบลปราสาท",
        "pages": 32
    },
    {
        "path": "referendum/inside/อำเภอเมืองบุรีรัมย์/ตำบลกระสัง",
        "pages": 28
    },
    {
        "path": "referendum/inside/อำเภอเมืองบุรีรัมย์/ตำบลกลันทา",
        "pages": 26
    },
    {
        "path": "referendum/inside/อำเภอเมืองบุรีรัมย์/ตำบลชุมเห็ด",
        "pages": 64
    },
    {
        "path": "referendum/inside/อำเภอเมืองบุรีรัมย์/ตำบลถลุงเหล็ก",
        "pages": 28
    },
    {
        "path": "referendum/inside/อำเภอเมืองบุรีรัมย์/ตำบลในเมือง",
        "pages": 62
    },
    {
        "path": "referendum/inside/อำเภอเมืองบุรีรัมย์/ตำบลบัวทอง",
        "pages": 28
    },
    {
        "path": "referendum/inside/อำเภอเมืองบุรีรัมย์/ตำบลบ้านบัว",
        "pages": 38
    },
    {
        "path": "referendum/inside/อำเภอเมืองบุรีรัมย์/ตำบลบ้านยาง",
        "pages": 44
    },
    {
        "path": "referendum/inside/อำเภอเมืองบุรีรัมย์/ตำบลพระครู",
        "pages": 22
    },
    {
        "path": "referendum/inside/อำเภอเมืองบุรีรัมย์/ตำบลลุมปุ๊ก",
        "pages": 38
    },
    {
        "path": "referendum/inside/อำเภอเมืองบุรีรัมย์/ตำบลสะแกโพรง",
        "pages": 48
    },
    {
        "path": "referendum/inside/อำเภอเมืองบุรีรัมย์/ตำบลหนองตาด",
        "pages": 44
    },
]

In [ ]:
PARTICIPANTS_DISTRICT = [
    [
        "นายภาคภูมิ โภคทรัพย์", 
        "นายพีรภัทร ทองธีรสกุล", 
        "นายธนายุทธ ยืนยั่ง", 
        "นางสุทธิลักษณ์ ยายิรัมย์", 
        "นายสนอง เทพอักษรณรงค์", 
        "นายวิเชียร ลานทอง", 
        "นายนาท ฉัพพรรณธนกูร", 
        "นายอิทธิพัทธ์ ภักดีเนติพันธุ์"
    ],
    
    [
        "ประชาธิปัตย์", 
        "เพื่อไทย", 
        "ประชาชน", 
        "รวมไทยสร้างชาติ", 
        "ภูมิใจไทย", 
        "ประชากรไทย", 
        "กล้าธรรม", 
        "เศรษฐกิจ"
    ],
]

In [ ]:
PARTYLIST_PAGE_I_1 = 10
PARTYLIST_PAGE_I_2 = 24
PARTYLIST_PAGE_I_3 = 23

PARTICIPANTS_PARTYLIST = [
    "ไทยทรัพย์ทวี", "เพื่อชาติไทย", "ใหม่",
    "มิติใหม่", "รวมใจไทย", "รวมไทยสร้างชาติ",
    "พลวัต", "ประชาธิปไตยใหม่", "เพื่อไทย",
    "ทางเลือกใหม่",

    "เศรษฐกิจ", "เสรีรวมไทย", "รวมพลังประชาชน", 
    "ท้องที่ไทย", "อนาคตไทย", "พลังเพื่อไทย", 
    "ไทยชนะ", "พลังสังคมใหม่", "สังคมประชาธิปไตยไทย", 
    "ฟิวชัน", "ไทรวมพลัง", "ก้าวอิสระ", 
    "ปวงชนไทย", "วิชชั่นใหม่", "เพื่อชีวิตใหม่", 
    "คลองไทย", "ประชาธิปัตย์", "ไทยก้าวหน้า", 
    "ไทยภักดี", "แรงงานสร้างชาติ", "ประชากรไทย", 
    "ครูไทยเพื่อประชาชน", "ประชาชาติ", "สร้างอนาคตไทย",

    "รักชาติ", "ไทยพร้อม",
    "ภูมิใจไทย", "พลังธรรมใหม่", "กรีน",
    "ไทยธรรม", "แผ่นดินธรรม", "กล้าธรรม",
    "พลังประชารัฐ", "โอกาสใหม่", "เป็นธรรม",
    "ประชาชน", "ประชาไทย", "ไทยสร้างไทย",
    "ไทยก้าวใหม่", "ประชาอาสาชาติ", "พร้อม",
    "เครือข่ายชาวนาแห่งประเทศไทย", "ไทยพิทักษ์ธรรม", "ความหวังใหม่",
    "ไทยรวมไทย", "เพื่อบ้านเมือง", "พลังไทยรักชาติ",
]

In [ ]:
REFERENDUM_LIST = ["เห็นชอบ", "ไม่เห็นชอบ", "ไม่แสดงความคิดเห็น"]

# OCR

In [ ]:
API_KEY = os.getenv("TYPHOON_OCR_API_KEY")

### Reader Function

In [ ]:
# อ่านข้อมูล file ตระกูลสส.เขต
def read_district(file_name: str, page_num: int):
    print(f"Processing District - File: {file_name} - Page: {page_num}")
    data_json = ocr_document(
        api_key=API_KEY,
        pdf_or_image_path=f"data/{file_name}.pdf",
        page_num=page_num,
        target_image_dim=2400,
    )
    
    data_json = thai_to_arabic(data_json)
    
    all_card = re.search(
        r'(?:ได้รับบัตรเลือกตั้ง.*?ทั้งหมด|จำนวนบัตรเลือกตั้งที่ได้รับจัดสรร.*?จำนวน)\s*(\d[\d,]*)\s*บัตร',
        data_json
    )
    good_card = re.search(r'บัตรดี.*?(\d[\d,]*)\s*บัตร', data_json)
    bad_card = re.search(r'บัตรเสีย.*?(\d[\d,]*)\s*บัตร', data_json)
    none_card = re.search(r'บัตรที่ไม่เลือกผู้สมัครผู้ใด.*?(\d[\d,]*)\s*บัตร', data_json)

    summary = {
        'จำนวนบัตรทั้งหมด': parse_number(all_card.group(1)) if all_card else 0,
        'บัตรดี': parse_number(good_card.group(1)) if good_card else 0,
        'บัตรเสีย': parse_number(bad_card.group(1)) if bad_card else 0,
        'ไม่เลือกผู้ใด': parse_number(none_card.group(1)) if none_card else 0,
    }
    
    df = pd.read_html(StringIO(data_json))[0]
        
    df.columns = ['หมายเลข', 'ชื่อผู้สมัคร', 'พรรค', 'คะแนน']
    df = df.iloc[1:].reset_index(drop=True)
    df.drop(columns=['หมายเลข'], inplace=True)
    
    # ตัดแถวให้เหลือแค่พรรคที่มี
    df = df.head(len(PARTICIPANTS_DISTRICT[0]))
    
    # ถ้าค่าใน column พรรค เป็นตัวเลข ให้เอามาใส่แทนที่ใน column คะแนน
    df[['พรรค', 'คะแนน']] = df.apply(swap_if_party_is_numeric, axis=1)[['พรรค', 'คะแนน']]
    
    # เติมชื่อผู้สมัครและพรรค
    df["ชื่อผู้สมัคร"] = PARTICIPANTS_DISTRICT[0]
    df["พรรค"] = PARTICIPANTS_DISTRICT[1]

    # Extract เฉพาะตัวเลขจากคะแนน
    df['คะแนน'] = (df['คะแนน']
                    .astype(str)
                    .str.replace(',', '')
                    .str.findall(r'\d+') 
                    .str[-1]
                    .pipe(pd.to_numeric, errors='coerce')
                    .fillna(0)
                    .astype(int))
    
    # เก็บผลลัพธ์
    save_ocr_output(file_name, page_num, summary, df)
    
    return summary, df

In [ ]:
# อ่านข้อมูล file ตระกูลสส.บัญชีรายชื่อ
def read_partylist(file_name: str, page_num: int):
    print(f"Processing Partylist - File: {file_name} - Page: {page_num}-{page_num+2}")
    
    # Page i
    data_json_page_i_1 = ocr_document(
        api_key=API_KEY,
        pdf_or_image_path=f"data/{file_name}.pdf",
        page_num=page_num,
        target_image_dim=2400,
    )
    
    # Page i+1
    data_json_page_i_2 = ocr_document(
        api_key=API_KEY,
        pdf_or_image_path=f"data/{file_name}.pdf",
        page_num=page_num+1,
        target_image_dim=2400,
    )
    
    # Page i+2
    data_json_page_i_3 = ocr_document(
        api_key=API_KEY,
        pdf_or_image_path=f"data/{file_name}.pdf",
        page_num=page_num+2,
        target_image_dim=2400,
    )
    
    # print(f"Page {page_num}")
    data_json_page_i_1 = thai_to_arabic(data_json_page_i_1)
    df_page_i_1 = pd.read_html(StringIO(data_json_page_i_1))[0]
    
    # print(f"Page {page_num+1}")
    data_json_page_i_2 = thai_to_arabic(data_json_page_i_2)
    df_page_i_2 = pd.read_html(StringIO(data_json_page_i_2))[0]
    
    # print(f"Page {page_num+2}")
    data_json_page_i_3 = thai_to_arabic(data_json_page_i_3)
    df_page_i_3 = pd.read_html(StringIO(data_json_page_i_3))[0]

    all_card = re.search(
        r'(?:ได้รับบัตรเลือกตั้ง.*?ทั้งหมด|จำนวนบัตรเลือกตั้งที่ได้รับจัดสรร.*?จำนวน)\s*(\d[\d,]*)\s*บัตร',
        data_json_page_i_1
    )
    good_card = re.search(r'บัตรดี.*?(\d[\d,]*)\s*บัตร', data_json_page_i_1)
    bad_card = re.search(r'บัตรเสีย.*?(\d[\d,]*)\s*บัตร', data_json_page_i_1)
    none_card = re.search(r'บัตรที่ไม่เลือกบัญชีรายชื่อของพรรคการเมืองใด.*?(\d[\d,]*)\s*บัตร', data_json_page_i_1)

    summary = {
        'จำนวนบัตรทั้งหมด': parse_number(all_card.group(1)) if all_card else 0,
        'บัตรดี': parse_number(good_card.group(1)) if good_card else 0,
        'บัตรเสีย': parse_number(bad_card.group(1)) if bad_card else 0,
        'ไม่เลือกผู้ใด': parse_number(none_card.group(1)) if none_card else 0,
    }
    
    df_page_i_1 = df_page_i_1.iloc[1:PARTYLIST_PAGE_I_1+1, :3].reset_index(drop=True)
    df_page_i_2 = df_page_i_2.iloc[1:PARTYLIST_PAGE_I_2+1, :3].reset_index(drop=True)
    df_page_i_3 = df_page_i_3.iloc[1:PARTYLIST_PAGE_I_3+1, :3].reset_index(drop=True)
    
    df = pd.concat([df_page_i_1, df_page_i_2, df_page_i_3])
    
    df.columns = ['หมายเลข', 'พรรค', 'คะแนน']
    df.drop(columns=['หมายเลข'], inplace=True)
    df.reset_index(drop=True)
    
    # เติมชื่อพรรค
    df["พรรค"] = PARTICIPANTS_PARTYLIST
    
    # Extract เฉพาะตัวเลขจากคะแนน
    df['คะแนน'] = (df['คะแนน']
                    .astype(str)
                    .str.replace(',', '')
                    .str.findall(r'\d+') 
                    .str[-1]
                    .pipe(pd.to_numeric, errors='coerce')
                    .fillna(0)
                    .astype(int))
    
    # เก็บผลลัพธ์
    save_ocr_output(file_name, page_num, summary, df)
    
    return summary, df

In [ ]:
# อ่านข้อมูล file ตระกูลประชามติ
def read_referendum(file_name: str, page_num: int):
    print(f"Processing Referendum - File: {file_name} - Page: {page_num}-{page_num+1}")
    
    # Page i
    data_json_page_i_1 = ocr_document(
        api_key=API_KEY,
        pdf_or_image_path=f"data/{file_name}.pdf",
        page_num=page_num,
        target_image_dim=2400,
    )
    
    # Page i+1
    data_json_page_i_2 = ocr_document(
        api_key=API_KEY,
        pdf_or_image_path=f"data/{file_name}.pdf",
        page_num=page_num+1,
        target_image_dim=2400,
    )

    # print(f"Page {page_num}")
    data_json_page_i_1 = thai_to_arabic(data_json_page_i_1)
    
    # print(f"Page {page_num+1}")
    data_json_page_i_2 = thai_to_arabic(data_json_page_i_2)
    df_page_i_2 = pd.read_html(StringIO(data_json_page_i_2))[0]
    
    df = pd.concat([df_page_i_2])
    
    print(df)
    
    total_eligible = re.search(
        r'ผู้มีสิทธิออกเสียงประชามติ.*?(\d[\d,]*)\s*คน',
        data_json_page_i_1
    )
    
    total_voted = re.search(
        r'มาแสดงตน.*?(\d[\d,]*)\s*คน',
        data_json_page_i_1
    )

    summary = {
        'ผู้มีสิทธิ': parse_number(total_eligible.group(1)) if total_eligible else 0,
        'มาใช้สิทธิ': parse_number(total_voted.group(1)) if total_voted else 0,
    }

    df = df.iloc[:, [2]]
    df.columns = ['คะแนน']
    df = df.dropna().reset_index(drop=True)

    # Extract เฉพาะตัวเลขจากคะแนน
    df['คะแนน'] = (df['คะแนน']
                    .astype(str)
                    .str.replace(',', '')
                    .str.findall(r'\d+')
                    .str[-1]
                    .pipe(pd.to_numeric, errors='coerce')
                    .fillna(0)
                    .astype(int)
    )

    # เติมรายการ
    df.insert(0, 'รายการ', REFERENDUM_LIST)
 
    save_ocr_output(file_name, page_num, summary, df)

    return summary, df

### Reader

In [ ]:
for file_config in files:
    total_pages = file_config["pages"]
    if "district" in file_config["path"]:
        for page_num in range(1, total_pages, 2):
            read_district(file_config["path"], page_num)
    elif "partylist" in file_config["path"]:
        for page_num in range(1, total_pages, 4):
            read_partylist(file_config["path"], page_num)
    elif "referendum" in file_config["path"]:
        for page_num in range(1, total_pages, 2):
            read_referendum(file_config["path"], page_num)

# Load Data

In [ ]:
dfs_district = []
dfs_partylist = []
dfs_referendum = []

for i in range(len(files)):
    file_config = files[i]
    total_pages = file_config["pages"]
    if "district" in file_config["path"]: 
        for page_num in range(1, total_pages, 2):
            df_dict = load_data(file_config["path"], page_num)
            dfs_district.append(df_dict)
            
    elif "partylist" in file_config["path"]:
        for page_num in range(1, total_pages, 4):
            df_dict = load_data(file_config["path"], page_num)
            dfs_partylist.append(df_dict)
            
    elif "referendum" in file_config["path"]:
        for page_num in range(1, total_pages, 2):
            df_dict = load_data(file_config["path"], page_num)
            dfs_referendum.append(df_dict)

In [ ]:
dfs_district[0]

In [ ]:
dfs_partylist[0]

In [ ]:
dfs_referendum[0]